# 08 — Run the best possession pipeline

This is the organiser-facing executable notebook. It runs the canonical `run_interpolated_flight_experiment.py` pipeline and writes the best observation-only candidate: visible-ball low-motion dead-ball evidence plus interval-level NO_SWITCH retention. Evaluation and disagreement analysis live in `08_model_evaluation.ipynb`.

The pipeline is observation-first: engineered observations → scenario evidence → deterministic anchors → bounded transition hypotheses → final H/A/D/X labels. Kalman estimates missing-ball geometry only; it does not decide possession. Viterbi/HMM/HSMM remain optional decoder diagnostics, not the primary truth-maker.

The runner also emits transition-signal ablations for notebook 09: post-contact/team-level true-switch confirmation, restart activation, their composition, a broad restart-phase interval resolver, and local restart-boundary guards. The resolver is the structural path we are testing: restart observations -> restart phase/evidence -> transition constraints -> shared interval resolver -> final H/A/D/X label. These candidates use observations only and are intentionally diagnostic; the best output is not replaced unless the common evaluator shows a real improvement in macro-F1, spell-F1, and transition F1@25.

George's optional fused ball observations enter earlier, in the shared observation table, through `possession_lab/run_george_ballfill_experiment.py`. They preserve the tactical track, add source/uncertainty evidence, and are evaluated as a separate ablation; they never become direct possession labels.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUNNER = ROOT / 'possession_lab' / 'run_interpolated_flight_experiment.py'
RUNNER_MODULE = 'possession_lab.run_interpolated_flight_experiment'
EVALUATOR = ROOT / 'evaluation' / 'run_evaluation.py'
RESULTS = ROOT / 'evaluation' / 'results'
REPORTS = ROOT / 'evaluation' / 'reports'
BEST_NAME = 'Visible-ball dead-ball + NO_SWITCH interval resolver'
BEST_OUTPUT = RESULTS / 'diagnostic_visible_ball_no_switch_interval.csv'
configured_data_root = os.environ.get('MTAH_DATA_DIR')
data_candidates = [Path(configured_data_root)] if configured_data_root else []
data_candidates += [ROOT / 'data', ROOT.parent / 'barca-hack-repo-staging' / 'data']
DATA_ROOT = next((path for path in data_candidates if (path / 'tactical' / 'BAR-ATH_tactical_atd.txt').exists()), None)
assert RUNNER.exists(), RUNNER
assert EVALUATOR.exists(), EVALUATOR
assert DATA_ROOT is not None, f'tactical data not found in: {data_candidates}'
print('project root:', ROOT)
print('runner:', RUNNER)
print('data root:', DATA_ROOT)
print('candidate:', BEST_NAME)

project root: /Users/luke/Desktop/barca-hack-repo-staging
runner: /Users/luke/Desktop/barca-hack-repo-staging/possession_lab/run_interpolated_flight_experiment.py
data root: /Users/luke/Desktop/barca-hack-repo-staging/data
candidate: Visible-ball dead-ball + NO_SWITCH interval resolver


## Execute the candidate pipeline

This cell regenerates the frame-aligned candidate and its diagnostic evidence files. No ground-truth labels are read by the runner.

In [2]:
runner_environment = os.environ.copy()
runner_environment['MTAH_DATA_DIR'] = str(DATA_ROOT)
pipeline = subprocess.run(
    [sys.executable, '-m', RUNNER_MODULE],
    cwd=ROOT,
    env=runner_environment,
    capture_output=True,
    text=True,
    check=False,
)
if pipeline.stdout:
    print(pipeline.stdout)
if pipeline.stderr:
    print('--- runner stderr ---')
    print(pipeline.stderr)
if pipeline.returncode != 0:
    raise RuntimeError(f'pipeline failed with exit code {pipeline.returncode}; see runner stderr above')
print('pipeline completed')
assert BEST_OUTPUT.exists(), BEST_OUTPUT
print('candidate written:', BEST_OUTPUT)

/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_scenario_bidirectional_interpolated_flight_ensemble.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_missing_ball_trajectory.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_kalman_missing_ball_trajectory.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_team_coordinate_path_bridge.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_same_player_dribble_gap_bridge.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_restart_type_specific_interpolated_flight.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/baseline_dead_ball_boundary_low_motion.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/diagnostic_visible_ball_low_motion_dead_ball.csv
/Users/luke/Desktop/barca-hack-repo-staging/evaluation/results/diagnostic_visible_ball_no_swi

## Outputs to present

- Candidate submission: `evaluation/results/diagnostic_visible_ball_no_switch_interval.csv`
- Scenario/trust evidence: `evaluation/reports/scenario_confidence_scores.csv`
- Transition evidence: `evaluation/reports/transition_hypothesis_ledger.csv`
- Transition-signal diagnostics: the runner also writes direct, hysteretic-handoff, restart-state, broad restart-phase, and local-boundary guard candidates under `evaluation/results/diagnostic_no_switch_*.csv`; they are ranked in notebook 09 and are not automatically promoted.
- Structural full-match candidate: `evaluation/results/diagnostic_no_switch_restart_phase_interval.csv` applies restart evidence as a constraint layer without allowing restart context to invent possession.
- Optional George ablation: run `python -m possession_lab.run_george_ballfill_experiment --ballfill /path/to/BAR-ATH_ball.csv` to produce `evaluation/results/diagnostic_george_fused_observed_no_switch_interval.csv`; this is a separate observation experiment, not the promoted candidate.

Run `08_model_evaluation.ipynb` next for public scoring and disagreement plots.